# Практическое занятие: Современная токенизация подслов (Subword Tokenization)
## Кейс: Сравнение BPE, Byte-Level BPE, WordPiece и SentencePiece на тексте Л.Н. Толстого

В современных трансформерных моделях (BERT, GPT, LLaMA) текст больше не делят строго по словам, так как это приводит к проблеме **OOV (Out-of-Vocabulary)** слов. Вместо этого используются алгоритмы, которые разбивают редкие и сложные слова на часто встречающиеся **подслова** (subwords) или байты.

На этом занятии мы сравним на одной фразе три ключевых алгоритма токенизации:
1. **Byte-Level BPE (BBPE)** — токенизация на уровне байтов, полностью исключающая токен `[UNK]` (на примере GPT-2/GPT-4).
2. **WordPiece** — алгоритм, максимизирующий правдоподобие языковой модели (на примере BERT / RuBERT).
3. **SentencePiece** — подход, рассматривающий пробел как полноценный символ, работающий без предварительного разделения по пробелам (на примере T5 / LLaMA).


In [1]:
%pip install --upgrade pip setuptools wheel
%pip install --upgrade transformers tokenizers protobuf sentencepiece

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached wheel-0.48.0-py3-none-any.whl.metadata (2.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.2 MB/s  0:00:00 eta 0:00:01
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
Using cached wheel-0.48.0-py3-none-any.whl (33 kB)
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 83.0.0
    Uninstalling setuptools-83.0.0:
      Successfully uninstalled setuptools-83.0.0━━━━━━━━━━━━━━━━━━ 1/3 [setuptools]
  Attempting uninstall: pip90m╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [setuptools]
    Found existing installation: pip 26.2━━━━━━━━━━━━━━━━━━━━━ 1/3 [setuptools]
    Uninstalling pip-26.2:╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [setuptools]
      Successfully uninstalled pip-26.2━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [setuptools]
   ━━━━━━━━━

In [2]:
from transformers import AutoTokenizer

# Берем фразу со сложными русскими словоформами из нашего текста Войны и Мира
sample_text = (
    "Над ним не было ничего уже, кроме неба,— высокого неба, не ясного, но все-таки неизмеримо высокого, "
    "с тихо ползущими по нем серыми облаками."
)

print("=== ИСХОДНЫЙ ТЕКСТ ===")
print(sample_text)


/Users/ksrve/WORKSPACE/ПРЕПОДАВАНИЕ/Обработка естественного языка/Мои лекции/Jupiter Notebooks/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== ИСХОДНЫЙ ТЕКСТ ===
Над ним не было ничего уже, кроме неба,— высокого неба, не ясного, но все-таки неизмеримо высокого, с тихо ползущими по нем серыми облаками.


## 1. WordPiece (Модель: RuBERT от DeepPavlov)
**WordPiece** сначала делает грубую токенизацию по пробелам, а затем разбивает слова. 
Для обозначения подслов, которые находятся в середине или конце слова (не являются началом), он использует специальный префикс **`##`**. Если токенизатор встречает незнакомый символ, он заменяет его на `[UNK]`.


In [3]:
print("=== 1. WORDPIECE (RuBERT) ===")

# Загружаем классический русский BERT
wordpiece_tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

# Токенизируем
wp_tokens = wordpiece_tokenizer.tokenize(sample_text)
print("Токены:")
print(wp_tokens)

# Посмотрим на числовые ID этих токенов
wp_ids = wordpiece_tokenizer.convert_tokens_to_ids(wp_tokens)
print(f"\nКоличество токенов: {len(wp_tokens)}")


=== 1. WORDPIECE (RuBERT) ===


Токены:
['Над', 'ним', 'не', 'было', 'ничего', 'уже', ',', 'кроме', 'неба', ',', '—', 'высокого', 'неба', ',', 'не', 'ясно', '##го', ',', 'но', 'все', '-', 'таки', 'неиз', '##мер', '##имо', 'высокого', ',', 'с', 'тихо', 'полз', '##ущими', 'по', 'нем', 'серыми', 'облаками', '.']

Количество токенов: 36


## 2. Byte-Level BPE / BBPE (Модель: rorbe-roberta-ru-base или аналогичные BPE для RU)
**Byte-Level BPE** переводит текст в байты (UTF-8), а затем объединяет их. Он не использует префикс `##`. Вместо этого пробел перед словом кодируется специальным символом (в зависимости от реализации, например, символом `Ġ` в токенизаторах на базе GPT/RoBERTa). 
**Главный плюс:** Базовый словарь состоит из 256 байт, поэтому модель способна закодировать абсолютно любой символ в мире без `[UNK]`.


In [4]:
print("=== 2.1 BYTE-LEVEL BPE (RoBERTa / GPT-стиль) ===")

# Используем токенизатор модели sberbank-ai/ruRoberta-large (архитектура BBPE)
bbpe_tokenizer = AutoTokenizer.from_pretrained("sberbank-ai/ruRoberta-large")

print("Исходных текст:")
print(sample_text)

print()

# Токенизируем
bbpe_tokens = bbpe_tokenizer.tokenize(sample_text)
print("Токены (символ Ġ означает пробел НАЧАЛА слова):")
print(bbpe_tokens)

print(f"\nКоличество токенов: {len(bbpe_tokens)}")


=== 2.1 BYTE-LEVEL BPE (RoBERTa / GPT-стиль) ===
Исходных текст:
Над ним не было ничего уже, кроме неба,— высокого неба, не ясного, но все-таки неизмеримо высокого, с тихо ползущими по нем серыми облаками.

Токены (символ Ġ означает пробел НАЧАЛА слова):
['ÐĿÐ°Ð´', 'ĠÐ½Ð¸Ð¼', 'ĠÐ½Ðµ', 'ĠÐ±ÑĭÐ»Ð¾', 'ĠÐ½Ð¸ÑĩÐµÐ³Ð¾', 'ĠÑĥÐ¶Ðµ', ',', 'ĠÐºÑĢÐ¾Ð¼Ðµ', 'ĠÐ½ÐµÐ±Ð°', ',âĢĶ', 'ĠÐ²ÑĭÑģÐ¾ÐºÐ¾Ð³Ð¾', 'ĠÐ½ÐµÐ±Ð°', ',', 'ĠÐ½Ðµ', 'ĠÑıÑģÐ½Ð¾Ð³Ð¾', ',', 'ĠÐ½Ð¾', 'ĠÐ²ÑģÐµ', '-', 'ÑĤÐ°ÐºÐ¸', 'ĠÐ½ÐµÐ¸Ð·Ð¼ÐµÑĢ', 'Ð¸Ð¼Ð¾', 'ĠÐ²ÑĭÑģÐ¾ÐºÐ¾Ð³Ð¾', ',', 'ĠÑģ', 'ĠÑĤÐ¸ÑħÐ¾', 'ĠÐ¿Ð¾Ð»Ð·', 'ÑĥÑīÐ¸Ð¼Ð¸', 'ĠÐ¿Ð¾', 'ĠÐ½ÐµÐ¼', 'ĠÑģÐµÑĢÑĭÐ¼Ð¸', 'ĠÐ¾Ð±Ð»Ð°ÐºÐ°Ð¼Ð¸', '.']

Количество токенов: 33


## 3. SentencePiece (Модель: RUT5 или mBART)
**SentencePiece** — это не просто алгоритм, а целая концепция. Он **НЕ делает предварительной токенизации по пробелам**. Он заменяет все пробелы специальным символом нижнего подчеркивания **`_`** (или ` `) и обрабатывает весь текст как единую строку. Это позволяет применять его к языкам, где пробелов вообще нет (например, китайский или японский). Под капотом SentencePiece может использовать как BPE, так и Unigram-модель.


In [5]:
print("=== 3. SENTENCEPIECE (RUT5 / mBART) ===")

# Используем токенизатор русскоязычной модели T5 (ai-forever/ruT5-base)
sp_tokenizer = AutoTokenizer.from_pretrained("ai-forever/ruT5-base")

# Токенизируем
sp_tokens = sp_tokenizer.tokenize(sample_text)
print("Токены (символ   означает пробел):")
print(sp_tokens)

print(f"\nКоличество токенов: {len(sp_tokens)}")


=== 3. SENTENCEPIECE (RUT5 / mBART) ===
Токены (символ   означает пробел):
['▁Над', '▁ним', '▁не', '▁было', '▁ничего', '▁уже', ',', '▁кроме', '▁неба', ',', '—', '▁высокого', '▁неба', ',', '▁не', '▁ясно', 'го', ',', '▁но', '▁все', '-', 'таки', '▁не', 'из', 'мер', 'имо', '▁высокого', ',', '▁с', '▁тихо', '▁полз', 'у', 'щими', '▁по', '▁нем', '▁серым', 'и', '▁облака', 'ми', '.']

Количество токенов: 40


## 4. Unigram (Модель XLNet)

**Unigram** — это простой тип вероятностной языковой модели (изучали с вами на второй лекции), в которой каждый токен (слово или часть слова) рассматривается независимо от соседних элементов.

Подробный разбор алгоритма доступен на Hugging Face: https://huggingface.co/learn/llm-course/ru/chapter6/7

In [6]:
print("=== 4. UNIGRAM (XLNet) ===")

# Загружаем мультиязычный токенизатор XLNet, использующий алгоритм Unigram под капотом
unigram_tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased")

# Токенизируем
unigram_tokens = unigram_tokenizer.tokenize(sample_text)
print("Токены (символ _ означает пробел):")
print(unigram_tokens)

print(f"\nКоличество токенов: {len(unigram_tokens)}")

=== 4. UNIGRAM (XLNet) ===
Токены (символ _ означает пробел):
['▁', 'Н', 'а', 'д', '▁', 'н', 'и', 'м', '▁', 'н', 'е', '▁', 'был', 'о', '▁', 'н', 'и', 'ч', 'е', 'г', 'о', '▁', 'уж', 'е', ',', '▁', 'к', 'р', 'о', 'м', 'е', '▁', 'н', 'е', 'б', 'а', ',', '—', '▁', 'в', 'ыс', 'о', 'к', 'о', 'г', 'о', '▁', 'н', 'е', 'б', 'а', ',', '▁', 'н', 'е', '▁', 'яс', 'н', 'о', 'г', 'о', ',', '▁', 'н', 'о', '▁', 'в', 'с', 'е', '-', 'т', 'а', 'к', 'и', '▁', 'н', 'е', 'и', 'зм', 'е', 'р', 'и', 'м', 'о', '▁', 'в', 'ыс', 'о', 'к', 'о', 'г', 'о', ',', '▁', 'с', '▁', 'т', 'и', 'х', 'о', '▁', 'п', 'о', 'лзущ', 'и', 'м', 'и', '▁', 'п', 'о', '▁', 'н', 'е', 'м', '▁', 'с', 'е', 'р', 'ым', 'и', '▁', 'о', 'бл', 'а', 'к', 'а', 'м', 'и', '.']

Количество токенов: 129


In [7]:
target_word = "неизмеримо"
print(f"Слово для анализа: '{target_word}'\n")

# 1. WordPiece
wp_word = wordpiece_tokenizer.tokenize(target_word)
print(f"WordPiece (RuBERT):       {wp_word}")

# 2. Byte-Level BPE
bbpe_word = bbpe_tokenizer.tokenize(target_word)
print(f"Byte-Level BPE (RoBERTa): {bbpe_word}")

# 3. SentencePiece
sp_word = sp_tokenizer.tokenize(target_word)
print(f"SentencePiece (ruT5):     {sp_word}")

# 4. Unigram
unigram_word = unigram_tokenizer.tokenize(target_word)
print(f"Unigram (XLNet):          {unigram_word}")


Слово для анализа: 'неизмеримо'

WordPiece (RuBERT):       ['неиз', '##мер', '##имо']
Byte-Level BPE (RoBERTa): ['Ð½Ðµ', 'Ð¸Ð·', 'Ð¼ÐµÑĢ', 'Ð¸Ð¼Ð¾']
SentencePiece (ruT5):     ['▁не', 'из', 'мер', 'имо']
Unigram (XLNet):          ['▁', 'н', 'е', 'и', 'зм', 'е', 'р', 'и', 'м', 'о']
